<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
.katex-display {
  text-align: left !important;
  margin: 1rem 0 1.25rem 2.5rem !important;
}
.katex-display > .katex {
  text-align: left !important;
}
.katex-display > .katex > .katex-html {
  text-align: left !important;
}
</style>

## 삽입 휴리스틱 (Insertion Heuristic)

<strong>삽입 휴리스틱은 일부 도시의 방문 순서를 먼저 정한 뒤, 아직 포함되지 않은 도시의 방문 위치를 하나씩 정해 전체 경로를 완성하는 방법이다.</strong>

### 1 · 핵심 원리

- <strong>적용 대상:</strong> 새 요소를 기존 순서의 여러 위치에 삽입할 수 있는 문제.
- <strong>선택 규칙:</strong> 여기서는 비용 증가가 최소인 도시와 위치를 고르는 cheapest insertion을 사용한다.
- <strong>삽입 비용:</strong> 도시 $i,j$ 사이에 $v$를 넣을 때, 기존 연결 비용을 빼고 새 연결 비용을 더한다. $C_{ij}$는 도시 $i$와 $j$ 사이의 거리이다.

$$
\Delta=C_{iv}+C_{vj}-C_{ij}.
$$


<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
.katex-display {
  text-align: left !important;
  margin: 1rem 0 1.25rem 2.5rem !important;
}
.katex-display > .katex {
  text-align: left !important;
}
.katex-display > .katex > .katex-html {
  text-align: left !important;
}
</style>

### 2 · 적용 예시

- <strong>예시: 다섯 도시의 배송 경로</strong>
  - 도시 0–4를 한 번씩 방문하고 출발 도시 1로 복귀한다. 목적함수는 총거리이며, 고정된 대칭 거리는 코드의 `distance`에 주어진다.
  - 먼저 도시 1에서 0을 방문한 뒤 1로 돌아오는 순서 $(1,0,1)$를 정한다. 총거리는 4이며, 아직 포함되지 않은 도시 2·3·4의 방문 위치를 정해야 한다.
  - 아래는 도시 1과 0 사이에 각 후보를 추가하는 비교이다. 회색 점선의 거리 2를 빼고 주황색 두 구간의 거리를 더한다. 도시 배치는 실제 거리와 무관하다.

  - <strong>도시 2:</strong> 거리 증가 8 + 9 − 2 = 15.

  <div style="text-align: left; margin: 0.65rem 0 1.5rem 2.5rem;"><img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/insertion_city_2.svg" alt="도시 1과 0 사이에 도시 2의 방문을 추가한다. 기존 거리 2를 제거하고 8와 9를 더하므로 총거리는 15만큼 증가한다." width="440" style="display: block; width: 27.5em; max-width: 100%; height: auto; margin: 0;"></div>

  - <strong>도시 3:</strong> 거리 증가 5 + 10 − 2 = 13.

  <div style="text-align: left; margin: 0.65rem 0 1.5rem 2.5rem;"><img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/insertion_city_3.svg" alt="도시 1과 0 사이에 도시 3의 방문을 추가한다. 기존 거리 2를 제거하고 5와 10를 더하므로 총거리는 13만큼 증가한다." width="440" style="display: block; width: 27.5em; max-width: 100%; height: auto; margin: 0;"></div>

  - <strong>도시 4:</strong> 거리 증가 6 + 7 − 2 = 11. 가장 작으므로 도시 4를 선택한다.

  <div style="text-align: left; margin: 0.65rem 0 1.5rem 2.5rem;"><img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/insertion_cost.svg" alt="도시 1과 0 사이에 도시 4의 방문을 추가한다. 기존 거리 2를 제거하고 6와 7를 더하므로 총거리는 11만큼 증가한다." width="440" style="display: block; width: 27.5em; max-width: 100%; height: auto; margin: 0;"></div>

### 3 · 구현

- 모든 미방문 도시와 삽입 위치를 비교한다. 동률이면 도시 번호, 삽입 위치 순서로 선택한다.
- 선택한 도시를 삽입하고, 모든 도시를 포함하면 종료한다.


In [ ]:
distance = [
    [0, 2, 9, 10, 7],
    [2, 0, 8, 5, 6],
    [9, 8, 0, 4, 3],
    [10, 5, 4, 0, 1],
    [7, 6, 3, 1, 0],
]

def tour_cost(tour):
    total = 0
    for i in range(len(tour) - 1):
        a = tour[i]
        b = tour[i + 1]
        total += distance[a][b]
    return total

tour = [1, 0, 1]
unvisited = [2, 3, 4]
while len(unvisited) > 0:
    best_increase = float("inf")  # Infinity: any candidate is smaller.
    best_city = unvisited[0]
    best_position = 1

    for city in unvisited:
        for position in range(1, len(tour)):
            a = tour[position - 1]
            b = tour[position]
            added_distance = distance[a][city] + distance[city][b]
            removed_distance = distance[a][b]
            increase = added_distance - removed_distance

            if increase < best_increase:
                best_increase = increase
                best_city = city
                best_position = position

    tour.insert(best_position, best_city)
    unvisited.remove(best_city)

print("Route:", tour)
print("Distance:", tour_cost(tour))

<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
.katex-display {
  text-align: left !important;
  margin: 1rem 0 1.25rem 2.5rem !important;
}
.katex-display > .katex {
  text-align: left !important;
}
.katex-display > .katex > .katex-html {
  text-align: left !important;
}
</style>

- <strong>결과:</strong> `[1, 3, 4, 2, 0, 1]`, 총거리 20.
- <strong>구성 과정:</strong> 도시 4·3·2를 순서대로 삽입하며 거리는 $4\to15\to15\to20$이 된다. 도시 3의 삽입 비용은 $5+1-6=0$이다.
- <strong>한계:</strong> 이 예제에서는 최단거리와 같지만, 현재의 최소 증가량이 완성된 경로의 최단거리를 보장하지는 않는다.
